In [22]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [23]:
#stations
df1 = pd.read_csv("stations.csv")
print("Train shape:", df1.shape)
display(df1.head())

Train shape: (7769, 21)


,id,node_id,trading_name,brand_name,organisation_name,latitude,longitude,address_line_1,city,county,...,is_motorway,is_supermarket,is_temporarily_closed,is_permanently_closed,amenities,opening_times,fuel_types,first_seen,last_seen,updated_at
0,6555,564736daa53d8b21e355dbc25999c38a0d588096517893...,PGG LAKELAND,Shell Lakeland,NaN,54.127270,-3.214640,ABBEY ROAD,BARROW IN FURNESS,CUMBRIA,...,False,False,False,False,"[""adblue_packaged"",""car_wash"",""water_filling""]","{""usual_days"":{""friday"":{""open"":""00:00:00"",""cl...","[""B7_PREMIUM"",""B7_STANDARD"",""E10"",""E5""]",2026-02-10T00:46:45.150Z,2026-03-28T00:19:17.355Z,2026-03-28T00:19:17.355Z
1,6566,8a3200296c6388336bc50f288a1222e7835a75160eb6ef...,MFG BICKER BAR,Esso,NaN,52.931700,-0.156615,DONNINGTON ROAD,BICKER BOSTON,LINCOLNSHIRE,...,False,False,False,False,"[""adblue_packaged"",""adblue_pumps"",""car_wash"",""...","{""usual_days"":{""friday"":{""open"":""00:00:00"",""cl...","[""B7_PREMIUM"",""B7_STANDARD"",""E10"",""E5""]",2026-02-10T00:46:45.173Z,2026-03-28T00:19:17.355Z,2026-03-28T00:19:17.355Z
2,6573,5eb2a81e4690b72f46efc3e854b61ed59f96f0e4617ff0...,MFG LEES ROAD,Esso,NaN,53.542262,-2.094393,129-131 LEES ROAD,OLDHAM,GREATER MANCHESTER,...,False,False,False,False,"[""adblue_packaged"",""car_wash"",""water_filling""]","{""usual_days"":{""friday"":{""open"":""00:00:00"",""cl...","[""B7_PREMIUM"",""B7_STANDARD"",""E10"",""E5""]",2026-02-10T00:46:45.190Z,2026-03-28T00:19:17.355Z,2026-03-28T00:19:17.355Z
3,13,69f0aac1004905db40032ccb11721e9a07ac66e34fd567...,PGG EPSOM DOWNS,Texaco Epsom Downs,NaN,51.303440,-0.270420,3 HEADLEY PARADE,LANGLEY VALERD,SURREY,...,False,False,False,False,"[""adblue_packaged"",""water_filling""]","{""usual_days"":{""friday"":{""open"":""06:00:00"",""cl...","[""B7_STANDARD"",""E10"",""E5""]",2026-02-10T00:46:29.845Z,2026-03-28T00:19:17.355Z,2026-03-28T00:19:17.355Z
4,6017,7bfc982de010496d07a1096b49da09d7143ff1c339f807...,Long Lane Filling Station,Harvest Energy,NaN,51.466500,0.145696,"W J KING (GARAGES) LTD, 160-164, LONG LANE, BE...",BEXLEYHEATH,NaN,...,False,False,False,False,"[""adblue_packaged"",""water_filling""]","{""usual_days"":{""friday"":{""open"":""00:00:00"",""cl...","[""B7_STANDARD"",""E10"",""E5""]",2026-02-10T00:46:43.931Z,2026-03-28T00:19:17.355Z,2026-03-28T00:19:17.355Z


In [24]:
targel_col = "fuel_types"

id_like1 = [i for i in df1.columns if i.lower() in ["id","unnamed: 0"]]
print("Kolom ID yang akan di-drop", id_like1)

x1 = df1.drop(columns=id_like1+[targel_col], errors = "ignore")
y1 = df1[targel_col]

print("X shape:", x1.shape)
print("y shape:", y1.shape)

Kolom ID yang akan di-drop ['id']
X shape: (7769, 19)
y shape: (7769,)


In [25]:
df = pd.read_csv("price_history.csv")
print("Train shape:", df.shape)
display(df.head())

Train shape: (284648, 6)


,id,node_id,fuel_type,price_pence,recorded_at,source_updated_at
0,1,4882e3fee979cfefa29a8777ce57ca0ecea27b4f12ba7f...,E10,126.0,2026-02-07T21:40:01.511Z,2026-02-06T12:46:05.000Z
1,2,4882e3fee979cfefa29a8777ce57ca0ecea27b4f12ba7f...,B7_STANDARD,135.0,2026-02-07T21:40:01.511Z,2026-02-06T12:46:05.000Z
2,3,b739362af81acc9fec9eda6f155348125fa2d5c1772c96...,E10,127.0,2026-02-07T21:40:01.511Z,2026-02-01T11:04:45.000Z
3,4,b739362af81acc9fec9eda6f155348125fa2d5c1772c96...,B7_STANDARD,138.0,2026-02-07T21:40:01.511Z,2026-02-01T11:04:45.000Z
4,5,ae5b2f8f6979b8b2460ff52f85619dd0dfc6a10bc98ca1...,E5,146.0,2026-02-07T21:40:01.511Z,2026-02-06T13:57:50.000Z


In [26]:
targel_col = "price_pence"

id_like = [i for i in df.columns if i.lower() in ["id","unnamed: 0"]]
print("Kolom ID yang akan di-drop", id_like)

x = df.drop(columns=id_like+[targel_col], errors = "ignore")
y = df[targel_col]

print("X shape:", x.shape)
print("y shape:", y.shape)

Kolom ID yang akan di-drop ['id']
X shape: (284648, 4)
y shape: (284648,)


In [27]:
missing = x.isna().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0].head(15))

counts = y.value_counts().notna()
display(counts.to_frame("count"))
print("Target distribution:")
display((counts / counts.sum()).to_frame("ratio"))

Missing values:


,0


,count
price_pence,
169.90,True
149.90,True
159.90,True
145.90,True
139.90,True
...,...
158.69,True
184.68,True
148.68,True


Target distribution:


,ratio
price_pence,
169.90,0.001078
149.90,0.001078
159.90,0.001078
145.90,0.001078
139.90,0.001078
...,...
158.69,0.001078
184.68,0.001078
148.68,0.001078


In [28]:
valid_idx = y.notna()
x_clean = x[valid_idx]
y_clean = y[valid_idx]

print(f"Sebelum bersihkan: {x.shape[0]} baris")
print(f"Setelah bersihkan: {x_clean.shape[0]} baris")
print(f"Jumlah NaN yang dihapus: {x.shape[0] - x_clean.shape[0]}")

Sebelum bersihkan: 284648 baris
Setelah bersihkan: 284648 baris
Jumlah NaN yang dihapus: 0


In [29]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state= RANDOM_STATE, stratify = None)

print("Train:", x_train.shape, "| Val:", x_val.shape)
print("\nTrain ratio:\n", y_train.value_counts(normalize=True).round(4))
print("\nVal ratio:\n", y_val.value_counts(normalize=True).round(4))

Train: (227718, 4) | Val: (56930, 4)

Train ratio:
 price_pence
169.900    0.0295
149.900    0.0294
159.900    0.0262
145.900    0.0242
139.900    0.0239
            ...  
123.695    0.0000
209.000    0.0000
11.410     0.0000
193.100    0.0000
182.890    0.0000
Name: proportion, Length: 877, dtype: float64

Val ratio:
 price_pence
169.900    0.0302
149.900    0.0288
159.900    0.0264
145.900    0.0251
139.900    0.0228
            ...  
166.200    0.0000
146.400    0.0000
142.312    0.0000
161.600    0.0000
143.200    0.0000
Name: proportion, Length: 525, dtype: float64


In [30]:
def evaluate_clf(y_true, y_pred, title="Model"):
    print(f"=== {title} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, average="weighted", zero_division=0))
    print("F1       :", f1_score(y_true, y_pred, average="weighted", zero_division=0))
    print("\nReport:\n", classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    ConfusionMatrixDisplay(cm, display_labels=np.unique(y_true)).plot(values_format="d")
    plt.title(f"Confusion Matrix — {title}")
    plt.show()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_w": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_w": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_w": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

In [31]:
# TODO: identify numerical and categorical columns from X_train
num_cols = x_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = x_train.select_dtypes(exclude=["number"]).columns.tolist()


print("Bagian 1")
print("Numerical columns: (%d)"% len(num_cols), num_cols)
print("Categorical columns: (%d)"% len(cat_cols),cat_cols)


print("\nBagian2")
print("Numerical:", len(num_cols), "| Categorical:", len(cat_cols))
print("Sample num:", num_cols[:6])
print("Sample cat:", cat_cols[:6])

Bagian 1
Numerical columns: (0) []
Categorical columns: (4) ['node_id', 'fuel_type', 'recorded_at', 'source_updated_at']

Bagian2
Numerical: 0 | Categorical: 4
Sample num: []
Sample cat: ['node_id', 'fuel_type', 'recorded_at', 'source_updated_at']


In [32]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
numeric_pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [33]:
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
categorical_pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))])

In [34]:
preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])
preprocess

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 []),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['node_id', 'fuel_type', 'recorded_at',
                                  'source_updated_at'])])

In [35]:
pipe_lr = Pipeline([ ("preprocess", preprocess),("model", LogisticRegression(max_iter=1000))])

pipe_lr.fit(x_train, y_train)
pred_lr = pipe_lr.predict(x_val)
evaluate_clf(y_val, pred_lr, "Baseline (Most Frequent)")

ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.